# Subtractive Genomics Pipeline — *Klebsiella pneumoniae* HS11286

Identifies non-host, non-homologous drug-target candidates from the *K. pneumoniae* subsp. *pneumoniae* HS11286 proteome (RefSeq GCF_000240185.1), via BLAST-based subtraction against the human proteome and Swiss-Prot, followed by annotation-based prioritization. Outputs shown are from the actual executed run.

**Input:** `kp_nr.fasta` — the HS11286 proteome after CD-HIT redundancy removal (5,779 → 5,637 sequences; CD-HIT was run upstream, in WSL, and is not repeated here — `kp_nr.fasta` is uploaded already deduplicated).


## Setup

In [1]:
from google.colab import files
uploaded = files.upload()

In [2]:
!apt-get update
!apt-get install ncbi-blast+

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,870 kB]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [62.6 kB]
Get

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os

output_folder = '/content/drive/MyDrive/Subtractive_Genomics_KP_Results'
os.makedirs(output_folder, exist_ok=True)
print(f"Created folder: {output_folder}")

## Step 1 — BLAST against the human proteome (remove host homologs)

Download the reviewed human proteome from UniProt, build a BLAST database, and BLAST `kp_nr.fasta` against it. (The UniProt REST download needs a quoted URL and comes back gzip-compressed despite the `.fasta`-looking name — decompressing before `makeblastdb` is what actually made this work.)

In [5]:
!wget "https://rest.uniprot.org/uniprotkb/stream?compressed=true&format=fasta&query=organism_id:9606"

--2026-03-17 13:24:57--  https://rest.uniprot.org/uniprotkb/stream?compressed=true&format=fasta&query=organism_id:9606
Resolving rest.uniprot.org (rest.uniprot.org)... 193.62.193.81
Connecting to rest.uniprot.org (rest.uniprot.org)|193.62.193.81|:443... connected.
HTTP request sent, awaiting response... 200 
Length: unspecified [text/plain]
Saving to: ‘stream?compressed=true&format=fasta&query=organism_id:9606’

stream?compressed=t     [       <=>          ]  29.55M   675KB/s    in 48s     

2026-03-17 13:25:46 (631 KB/s) - ‘stream?compressed=true&format=fasta&query=organism_id:9606’ saved [30991306]



In [6]:
!mv "stream?compressed=true&format=fasta&query=organism_id:9606" human.fasta

In [7]:
!mv human.fasta human.fasta.gz
!gunzip human.fasta.gz

In [8]:
!makeblastdb -in human.fasta -dbtype prot -out human_db



Building a new DB, current time: 03/17/2026 13:31:12
New DB name:   /content/human_db
New DB title:  human.fasta
Sequence type: Protein
Keep MBits: T
Maximum file size: 1000000000B
Adding sequences from FASTA; added 205155 sequences in 5.78587 seconds.




In [9]:
!blastp -query kp_nr.fasta -db human_db -out kp_vs_human.txt -evalue 1e-3 -outfmt 6 -num_threads 2

In [10]:
!ls -lh

total 187M
-rw-r--r-- 1 root root  20K Mar 17 13:31 human_db.pdb
-rw-r--r-- 1 root root  33M Mar 17 13:31 human_db.phr
-rw-r--r-- 1 root root 1.6M Mar 17 13:31 human_db.pin
-rw-r--r-- 1 root root 2.4M Mar 17 13:31 human_db.pot
-rw-r--r-- 1 root root  60M Mar 17 13:31 human_db.psq
-rw-r--r-- 1 root root  16K Mar 17 13:31 human_db.ptf
-rw-r--r-- 1 root root 802K Mar 17 13:31 human_db.pto
-rw-r--r-- 1 root root  82M Mar 17 13:25 human.fasta
-rw-r--r-- 1 root root 2.1M Mar 17 13:10 kp_nr.fasta
-rw-r--r-- 1 root root 6.0M Mar 17 14:27 kp_vs_human.txt
drwxr-xr-x 1 root root 4.0K Jan 16 14:24 sample_data


In [11]:
import shutil

file_to_save = 'kp_vs_human.txt'
destination_path = os.path.join(output_folder, file_to_save)
shutil.copy(file_to_save, destination_path)
print(f"Copied '{file_to_save}' to '{destination_path}'")

In [12]:
import shutil
import os

output_folder = '/content/drive/MyDrive/Subtractive_Genomics_KP_Results'

file_to_save_2 = 'kp_nr.fasta'
destination_path_2 = os.path.join(output_folder, file_to_save_2)
shutil.copy(file_to_save_2, destination_path_2)
print(f"Copied '{file_to_save_2}' to '{destination_path_2}'")

## Step 2 — Extract non-human sequences

Parse the BLAST hits for the set of query IDs with a human match, then write every `kp_nr.fasta` sequence *not* in that set to `kp_non_human.fasta`.

In [13]:
matched_query_ids = set()

with open('kp_vs_human.txt', 'r') as f:
    for line in f:
        # BLAST output format 6 is tab-separated
        parts = line.strip().split('\t')
        if parts:
            # The first column is the query sequence ID
            query_id = parts[0]
            matched_query_ids.add(query_id)

print(f"Found {len(matched_query_ids)} unique query sequences with matches in human proteins.")

Found 1467 unique query sequences with matches in human proteins.


In [14]:
non_human_sequences_data = []
current_header = ""
current_sequence = []

with open('kp_nr.fasta', 'r') as infile:
    for line in infile:
        line = line.strip()
        if line.startswith('>'):
            # If we've collected a full sequence and its header
            if current_header and current_sequence:
                # Extract the ID from the header (e.g., '>YP_005220808.1 replication protein A' -> 'YP_005220808.1')
                # The ID is usually the first word after '>' and before the first space
                sequence_id = current_header[1:].split(' ')[0]
                if sequence_id not in matched_query_ids:
                    non_human_sequences_data.append(current_header)
                    non_human_sequences_data.append("".join(current_sequence))

            # Start new sequence: store new header and reset sequence collector
            current_header = line
            current_sequence = []
        else:
            # Append sequence lines
            current_sequence.append(line)

    # After the loop, process the last sequence in the file
    if current_header and current_sequence:
        sequence_id = current_header[1:].split(' ')[0]
        if sequence_id not in matched_query_ids:
            non_human_sequences_data.append(current_header)
            non_human_sequences_data.append("".join(current_sequence))

# Write the collected non-human homologous sequences to a new FASTA file
output_filename = 'kp_non_human.fasta'
with open(output_filename, 'w') as outfile:
    for item in non_human_sequences_data:
        outfile.write(item + '\n')

print(f"Non-human homologous sequences saved to '{output_filename}'")

Non-human homologous sequences saved to 'kp_non_human.fasta'


In [15]:
import shutil
import os

output_folder = '/content/drive/MyDrive/Subtractive_Genomics_KP_Results'

# Ensure the output folder exists (it was created earlier, but good practice to ensure)
os.makedirs(output_folder, exist_ok=True)

file_to_save = 'kp_non_human.fasta'
destination_path = os.path.join(output_folder, file_to_save)
shutil.copy(file_to_save, destination_path)
print(f"Copied '{file_to_save}' to '{destination_path}'")

Copied 'kp_non_human.fasta' to '/content/drive/MyDrive/Subtractive_Genomics_KP_Results/kp_non_human.fasta'


## Step 3 — BLAST against Swiss-Prot (remove well-characterized homologs)

Swiss-Prot was used here in place of the originally planned Database of Essential Genes (DEG) — the DEG mirror and a GitHub fallback both failed (shown below), so the pipeline pivoted to reviewed UniProt/Swiss-Prot instead. This step should **not** be read as an essentiality filter — it removes proteins with strong matches in a curated, well-characterized reference set, which is a different (and weaker) criterion than predicted essentiality.

In [16]:
!wget http://tubic.tju.edu.cn/deg_test/public/download/DEG10.aa.gz
!gunzip DEG10.aa.gz

--2026-03-18 10:12:55--  http://tubic.tju.edu.cn/deg_test/public/download/DEG10.aa.gz
Resolving tubic.tju.edu.cn (tubic.tju.edu.cn)... 202.113.2.198
Connecting to tubic.tju.edu.cn (tubic.tju.edu.cn)|202.113.2.198|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: http://tubic.org [following]
--2026-03-18 10:12:56--  http://tubic.org/
Resolving tubic.org (tubic.org)... 3.167.163.44, 3.167.163.92, 3.167.163.108, ...
Connecting to tubic.org (tubic.org)|3.167.163.44|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://tubic.org/ [following]
--2026-03-18 10:12:57--  https://tubic.org/
Connecting to tubic.org (tubic.org)|3.167.163.44|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 16441 (16K) [text/html]
Saving to: ‘DEG10.aa.gz’

DEG10.aa.gz         100%[===================>]  16.06K  --.-KB/s    in 0s      

2026-03-18 10:12:57 (176 MB/s) - ‘DEG10.aa.gz’ saved [16441/16441]


gzip: 

In [17]:
!wget https://raw.githubusercontent.com/biobootloader/essential-genes/main/deg_proteins.fasta

--2026-03-18 10:14:42--  https://raw.githubusercontent.com/biobootloader/essential-genes/main/deg_proteins.fasta
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-03-18 10:14:42 ERROR 404: Not Found.



In [18]:
!wget "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=reviewed:true" -O swissprot.fasta

--2026-03-19 05:18:00--  https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=reviewed:true
Resolving rest.uniprot.org (rest.uniprot.org)... 193.62.193.81
Connecting to rest.uniprot.org (rest.uniprot.org)|193.62.193.81|:443... connected.
HTTP request sent, awaiting response... 200 
Length: unspecified [text/plain]
Saving to: ‘swissprot.fasta’

swissprot.fasta         [                <=> ] 274.16M  2.96MB/s    in 1m 53s  

2026-03-19 05:19:54 (2.42 MB/s) - ‘swissprot.fasta’ saved [287478568]



In [19]:
!makeblastdb -in swissprot.fasta -dbtype prot -out swiss_db



Building a new DB, current time: 03/19/2026 05:20:16
New DB name:   /content/swiss_db
New DB title:  swissprot.fasta
Sequence type: Protein
Keep MBits: T
Maximum file size: 1000000000B
Adding sequences from FASTA; added 574627 sequences in 17.5247 seconds.




In [20]:
!blastp \
-query kp_non_human.fasta \
-db swiss_db \
-out kp_vs_swiss.txt \
-evalue 1e-5 \
-outfmt 6 \
-num_threads 2 \
-max_target_seqs 5

In [21]:
!ls -lh kp_vs_swiss.txt

-rw-r--r-- 1 root root 996K Mar 19 07:21 kp_vs_swiss.txt


In [22]:
import shutil
import os

output_folder = '/content/drive/MyDrive/Subtractive_Genomics_KP_Results'

# Ensure the output folder exists (it was created earlier, but good practice to ensure)
os.makedirs(output_folder, exist_ok=True)

file_to_save = 'kp_vs_swiss.txt'
destination_path = os.path.join(output_folder, file_to_save)
shutil.copy(file_to_save, destination_path)
print(f"Copied '{file_to_save}' to '{destination_path}'")

Copied 'kp_vs_swiss.txt' to '/content/drive/MyDrive/Subtractive_Genomics_KP_Results/kp_vs_swiss.txt'


## Step 4 — Extract the final non-homologous candidate set

Install `seqkit`, get the unique set of Swiss-Prot-matched IDs, and remove them from `kp_non_human.fasta` to get the final candidate set.

In [23]:
!apt-get install -y seqkit

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  seqkit
0 upgraded, 1 newly installed, 0 to remove and 5 not upgraded.
Need to get 6,544 kB of archives.
After this operation, 15.2 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 seqkit amd64 2.1.0+ds-1ubuntu0.1 [6,544 kB]
Fetched 6,544 kB in 1s (6,474 kB/s)
Selecting previously unselected package seqkit.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../seqkit_2.1.0+ds-1ubuntu0.1_amd64.deb ...
Unpacking seqkit (2.1.0+ds-1ubuntu0.1) ...
Setting up seqkit (2.1.0+ds-1ubuntu0.1) ...
Processing triggers for man-db (2.10.2-1) ...


In [24]:
!cut -f1 kp_vs_swiss.txt | sort | uniq > matched_ids.txt

In [25]:
!head matched_ids.txt
!wc -l matched_ids.txt

YP_005220808.1
YP_005220812.1
YP_005220814.1
YP_005220820.1
YP_005220821.1
YP_005220824.1
YP_005220829.1
YP_005220831.1
YP_005220832.1
YP_005220836.1
3111 matched_ids.txt


In [26]:
!seqkit grep -v -f matched_ids.txt kp_non_human.fasta > kp_final.fasta

[INFO] 3111 patterns loaded from file


In [27]:
!grep -c ">" kp_non_human.fasta
!grep -c ">" kp_final.fasta

4170
1059


In [28]:
!cp kp_final.fasta /content/drive/MyDrive/Subtractive_Genomics_KP_Results/

## Step 5 — Validation checks

Two sanity checks on the *original* (pre-subtraction) proteome:
- **Negative control:** NDM1 (a well-known, human-irrelevant carbapenemase) should not show up as a target of interest here — it's screened for separately, not part of this prioritization.
- **Positive control:** known resistance genes (beta-lactamases) should still be present in `kp_nr.fasta`, confirming the proteome file itself is intact and complete.

In [29]:
import os

search_term = 'NDM1'
input_fasta = '/content/drive/MyDrive/Subtractive_Genomics_KP_Results/kp_nr.fasta'
output_file = 'ndm1_search_results.txt'

# Construct the grep command to search for the term in header lines only
# -E for extended regex, -i for case-insensitive, ^> to match start of line with >, then search term
# Alternatively, we can use `grep ">" kp_nr.fasta | grep -i NDM1`
# For simplicity and directness, I'll search for the term within any line starting with >
# !grep -i "^>.*NDM1" kp_nr.fasta > ndm1_search_results.txt

# A more robust way using grep to ensure it's in the header part, case-insensitive
# grep -P is for Perl regular expressions which allows for lookaheads etc, but -E is sufficient for this pattern
!grep -E '^>.*($search_term|New Delhi metallo-beta-lactamase)' "$input_fasta" > "$output_file"

print(f"Searched '{search_term}' and 'New Delhi metallo-beta-lactamase' in headers of '{input_fasta}' and saved results to '{output_file}'.")

Searched 'NDM1' and 'New Delhi metallo-beta-lactamase' in headers of '/content/drive/MyDrive/Subtractive_Genomics_KP_Results/kp_nr.fasta' and saved results to 'ndm1_search_results.txt'.


In [30]:
!cat ndm1_search_results.txt
!wc -l ndm1_search_results.txt

0 ndm1_search_results.txt


In [31]:
!grep -iE ">.*(beta-lactamase|carbapenemase|bla)" /content/drive/MyDrive/Subtractive_Genomics_KP_Results/kp_nr.fasta

>YP_005221002.1 beta-lactamase (plasmid) [Klebsiella pneumoniae subsp. pneumoniae HS11286]
>YP_005221024.1 extended-spectrum beta-lactamase/aminoglycoside modifying enzyme fusion protein (plasmid) [Klebsiella pneumoniae subsp. pneumoniae HS11286]
>YP_005221079.1 beta-lactamase CTX-M-14 (plasmid) [Klebsiella pneumoniae subsp. pneumoniae HS11286]
>YP_005225355.1 beta-lactamase/D-alanine carboxypeptidase [Klebsiella pneumoniae subsp. pneumoniae HS11286]
>YP_005225435.1 beta-lactamase synthesis regulator/muropeptide transporter [Klebsiella pneumoniae subsp. pneumoniae HS11286]
>YP_005225688.1 beta-lactamase [Klebsiella pneumoniae subsp. pneumoniae HS11286]
>YP_005226822.1 beta-lactamase SHV-11 [Klebsiella pneumoniae subsp. pneumoniae HS11286]
>YP_005227289.1 beta-lactamase [Klebsiella pneumoniae subsp. pneumoniae HS11286]
>YP_005227986.1 putative beta-lactamase [Klebsiella pneumoniae subsp. pneumoniae HS11286]
>YP_005229652.1 Class A Carbapenemase Kpc-2 (plasmid) [Klebsiella pneumoniae sub

## Step 6 — Candidate annotation & prioritization

List the annotations of the 1,059 final candidates, drop plasmid-encoded and uncharacterized ("hypothetical protein") entries, then rank the remainder by virulence/transport-related keywords.

In [32]:
!cp /content/drive/MyDrive/Subtractive_Genomics_KP_Results/kp_final.fasta /content/
!grep -c ">" kp_final.fasta   # sanity check — should print 1059

1059


In [33]:
!grep ">" kp_final.fasta > kp_final_annotations.txt
!wc -l kp_final_annotations.txt
!head -20 kp_final_annotations.txt

1059 kp_final_annotations.txt
>YP_005220809.1 hypothetical protein (plasmid) [Klebsiella pneumoniae subsp. pneumoniae HS11286]
>YP_005220810.1 hypothetical protein (plasmid) [Klebsiella pneumoniae subsp. pneumoniae HS11286]
>YP_005220811.1 hypothetical protein (plasmid) [Klebsiella pneumoniae subsp. pneumoniae HS11286]
>YP_005220813.1 hypothetical protein (plasmid) [Klebsiella pneumoniae subsp. pneumoniae HS11286]
>YP_005220815.1 putative regulator (plasmid) [Klebsiella pneumoniae subsp. pneumoniae HS11286]
>YP_005220816.1 hypothetical protein (plasmid) [Klebsiella pneumoniae subsp. pneumoniae HS11286]
>YP_005220817.1 lipoprotein (plasmid) [Klebsiella pneumoniae subsp. pneumoniae HS11286]
>YP_005220818.1 hypothetical protein (plasmid) [Klebsiella pneumoniae subsp. pneumoniae HS11286]
>YP_005220819.1 hypothetical protein (plasmid) [Klebsiella pneumoniae subsp. pneumoniae HS11286]
>YP_005220823.1 putative porphyrin biosynthetic protein (plasmid) [Klebsiella pneumoniae subsp. pneumoniae H

In [34]:
plasmid_count = 0
chromosomal_count = 0
hypothetical_count = 0
named_count = 0

with open('kp_final_annotations.txt') as f:
    for line in f:
        line_lower = line.lower()
        if '(plasmid)' in line_lower:
            plasmid_count += 1
        else:
            chromosomal_count += 1
        if 'hypothetical protein' in line_lower:
            hypothetical_count += 1
        else:
            named_count += 1

print(f"Plasmid-encoded: {plasmid_count}")
print(f"Chromosomal: {chromosomal_count}")
print(f"Hypothetical: {hypothetical_count}")
print(f"Named/annotated: {named_count}")

Plasmid-encoded: 265
Chromosomal: 794
Hypothetical: 718
Named/annotated: 341


In [35]:
final_candidates = []

with open('kp_final_annotations.txt') as f:
    for line in f:
        line_stripped = line.strip()
        line_lower = line_stripped.lower()
        is_plasmid = '(plasmid)' in line_lower
        is_hypothetical = 'hypothetical protein' in line_lower
        if not is_plasmid and not is_hypothetical:
            final_candidates.append(line_stripped)

print(f"Chromosomal + named candidates: {len(final_candidates)}")

with open('kp_priority_candidates.txt', 'w') as out:
    out.write('\n'.join(final_candidates))

# preview
for line in final_candidates[:30]:
    print(line)

Chromosomal + named candidates: 296
>YP_005224336.1 putative 2-component transcriptional regulator [Klebsiella pneumoniae subsp. pneumoniae HS11286]
>YP_005224343.1 DUF1471 domain-containing protein
>YP_005224345.1 PTS family protein II component [Klebsiella pneumoniae subsp. pneumoniae HS11286]
>YP_005224350.1 putative serine protease [Klebsiella pneumoniae subsp. pneumoniae HS11286]
>YP_005224351.1 putative inner membrane protein [Klebsiella pneumoniae subsp. pneumoniae HS11286]
>YP_005224458.1 50S ribosomal protein L15 [Klebsiella pneumoniae subsp. pneumoniae HS11286]
>YP_005224549.1 GNAT family N-acetyltransferase
>YP_005224552.1 L-asparagine permease [Klebsiella pneumoniae subsp. pneumoniae HS11286]
>YP_005224572.1 hemolysin [Klebsiella pneumoniae subsp. pneumoniae HS11286]
>YP_005224590.1 prepilin peptidase dependent protein C [Klebsiella pneumoniae subsp. pneumoniae HS11286]
>YP_005224597.1 periplasmic maltose-binding protein [Klebsiella pneumoniae subsp. pneumoniae HS11286]
>YP

In [36]:
tier1_keywords = ['adhesin', 'fimbri', 'hemolysin', 'toxin', 'secretion', 'dsba', 'hemin', 'siderophore',
                   'iron', 'porin', 'efflux', 'permease', 'transporter', 'pts family', 'autotransporter',
                   'invasin', 'capsule', 'lipopolysaccharide', 'lps']
deprioritize_keywords = ['ribosomal', 'trna', 'aminoacyl', 'rna polymerase subunit', 'dna polymerase',
                          'elongation factor', 'initiation factor', 'chaperone', 'heat shock']
vague_keywords = ['putative cytoplasmic protein', 'putative inner membrane protein', 'domain-containing protein',
                   'duf', 'family protein']

tier1, tier2, tier3_deprioritize, tier4_vague = [], [], [], []

with open('kp_priority_candidates.txt') as f:
    for line in f:
        line_s = line.strip()
        line_l = line_s.lower()
        if any(k in line_l for k in tier1_keywords):
            tier1.append(line_s)
        elif any(k in line_l for k in deprioritize_keywords):
            tier3_deprioritize.append(line_s)
        elif any(k in line_l for k in vague_keywords):
            tier4_vague.append(line_s)
        else:
            tier2.append(line_s)

print(f"Tier 1 (virulence/transport/surface — strong candidates): {len(tier1)}")
print(f"Tier 2 (other named — worth reviewing): {len(tier2)}")
print(f"Tier 3 (housekeeping/translation — deprioritize): {len(tier3_deprioritize)}")
print(f"Tier 4 (vague/uncharacterized — deprioritize for now): {len(tier4_vague)}")

with open('kp_tier1_candidates.txt', 'w') as out:
    out.write('\n'.join(tier1))

Tier 1 (virulence/transport/surface — strong candidates): 47
Tier 2 (other named — worth reviewing): 174
Tier 3 (housekeeping/translation — deprioritize): 4
Tier 4 (vague/uncharacterized — deprioritize for now): 71


In [37]:
with open('kp_tier1_candidates.txt') as f:
    print(f.read())

>YP_005224345.1 PTS family protein II component [Klebsiella pneumoniae subsp. pneumoniae HS11286]
>YP_005224552.1 L-asparagine permease [Klebsiella pneumoniae subsp. pneumoniae HS11286]
>YP_005224572.1 hemolysin [Klebsiella pneumoniae subsp. pneumoniae HS11286]
>YP_005224615.1 putative adhesin [Klebsiella pneumoniae subsp. pneumoniae HS11286]
>YP_005224626.1 type IV toxin-antitoxin system AbiEi family antitoxin
>YP_005224627.1 nucleotidyl transferase AbiEii/AbiGii toxin family protein
>YP_005224660.1 putative DSBA oxidoreductase [Klebsiella pneumoniae subsp. pneumoniae HS11286]
>YP_005224661.1 hemin storage system HmsS protein [Klebsiella pneumoniae subsp. pneumoniae HS11286]
>YP_005224686.1 fimbrial protein
>YP_005224689.1 type 1 fimbrial protein [Klebsiella pneumoniae subsp. pneumoniae HS11286]
>YP_005224858.1 Hcp1 family type VI secretion system effector, partial [Klebsiella pneumoniae subsp. pneumoniae HS11286]
>YP_005225553.1 RelE family toxin-antitoxin system [Klebsiella pneumoni

In [38]:
import shutil
import os

output_folder = '/content/drive/MyDrive/Subtractive_Genomics_KP_Results'

# Ensure the output folder exists
os.makedirs(output_folder, exist_ok=True)

files_to_save = [
    'kp_final_annotations.txt',
    'kp_priority_candidates.txt',
    'kp_tier1_candidates.txt'
]

for file_name in files_to_save:
    destination_path = os.path.join(output_folder, file_name)
    if os.path.exists(file_name):
        shutil.copy(file_name, destination_path)
        print(f"Copied '{file_name}' to '{destination_path}'")
    else:
        print(f"File '{file_name}' not found. Skipping copy.")


Copied 'kp_final_annotations.txt' to '/content/drive/MyDrive/Subtractive_Genomics_KP_Results/kp_final_annotations.txt'
Copied 'kp_priority_candidates.txt' to '/content/drive/MyDrive/Subtractive_Genomics_KP_Results/kp_priority_candidates.txt'
Copied 'kp_tier1_candidates.txt' to '/content/drive/MyDrive/Subtractive_Genomics_KP_Results/kp_tier1_candidates.txt'


## Step 7 — Target extraction and validation

Three candidates were shortlisted from the Tier-1 list for closer review: **DsbA**, **Irp3**, and **ClpV1**. Extracting full sequences revealed ClpV1 was a truncated ORF fragment (72 aa, vs. the expected ~800–900 aa for this protein family) — a broken annotation, not a real candidate — so it was dropped.

In [39]:
target_ids = ['YP_005224660.1', 'YP_005227768.1', 'YP_005227286.1']

def extract_sequences(fasta_file, ids_wanted, output_file):
    write_flag = False
    with open(fasta_file) as infile, open(output_file, 'w') as outfile:
        for line in infile:
            if line.startswith('>'):
                seq_id = line[1:].split()[0]
                write_flag = seq_id in ids_wanted
            if write_flag:
                outfile.write(line)

extract_sequences('kp_final.fasta', target_ids, 'top3_targets.fasta')
print("Done — check top3_targets.fasta")

Done — check top3_targets.fasta


In [40]:
!cat top3_targets.fasta

>YP_005224660.1 putative DSBA oxidoreductase [Klebsiella pneumoniae subsp. pneumoniae HS11286]
MSGHKKYLSVALLSAVLLPAMASAADAPATFTPEQEAKIGKIAADYLVAHPEVLLQASQK
LQQIQAEQQASAATQAVLKNAAVLTQDKNTPTYGPANGKVTVIEFFDYQCVYCSRLAPVM
EQVIKAHPQTRFAFKEWPIFGGRWESSLEAAKTGLQIYQQKGADAYLAYHNGIYATGHNE
GKLTTADIQQQAKKAGFDAKKAADVEPVLQSINDLAQEIGLSGTPGVIVMPTTGATEASI
TVFPGLADKASLEAAIKKAGG
>YP_005227286.1 type VI secretion ATPase, ClpV1 family [Klebsiella pneumoniae subsp. pneumoniae HS11286]
MFSGGRNIDSLLNQQLLTHMAAGQKPQHLTRGGMRKRGTPQEECPLFLQEVLPDYYYPLE
LKHPEQYTRPDI
>YP_005227768.1 irp3 protein, yersiniabactin siderophore biosynthetic protein [Klebsiella pneumoniae subsp. pneumoniae HS11286]
MMPSASPKQRVLIVGAKFGEMYLNAFMQPPEGLELVGLLAQGSARSRELAHAFGIPLYTS
PEQITRMPDIACIVVRSTVAGGTGTQLARHFLTRGVHVIQEHPLHPDDISSLQTLAQEQG
CCYWVNTFYPHTRAGRTWLRDAQQLRRCLAKTPPVVHATTSRQLLYSTLDLLLLALGVDA
AAVECDVVGSFSDFHCLRLFWPEGEACLLLQRYLDPDDPDMHSLIMHRLLLGWPEGHLSL
EASYGPVIWSSSLFVADHQENAHSLYRRPEILRDLPGLTRSAAPLSWRDCCETVGPEGVS
WLLHQLRSHLAGEHPPAACQSVHQIAL

In [41]:
target_ids = ['YP_005224660.1', 'YP_005227768.1']  # ClpV1 dropped — truncated fragment

def extract_sequences(fasta_file, ids_wanted, output_file):
    write_flag = False
    with open(fasta_file) as infile, open(output_file, 'w') as outfile:
        for line in infile:
            if line.startswith('>'):
                seq_id = line[1:].split()[0]
                write_flag = seq_id in ids_wanted
            if write_flag:
                outfile.write(line)

extract_sequences('kp_final.fasta', target_ids, 'final_2_targets.fasta')
print("Done")

Done


In [42]:
!cat final_2_targets.fasta

>YP_005224660.1 putative DSBA oxidoreductase [Klebsiella pneumoniae subsp. pneumoniae HS11286]
MSGHKKYLSVALLSAVLLPAMASAADAPATFTPEQEAKIGKIAADYLVAHPEVLLQASQK
LQQIQAEQQASAATQAVLKNAAVLTQDKNTPTYGPANGKVTVIEFFDYQCVYCSRLAPVM
EQVIKAHPQTRFAFKEWPIFGGRWESSLEAAKTGLQIYQQKGADAYLAYHNGIYATGHNE
GKLTTADIQQQAKKAGFDAKKAADVEPVLQSINDLAQEIGLSGTPGVIVMPTTGATEASI
TVFPGLADKASLEAAIKKAGG
>YP_005227768.1 irp3 protein, yersiniabactin siderophore biosynthetic protein [Klebsiella pneumoniae subsp. pneumoniae HS11286]
MMPSASPKQRVLIVGAKFGEMYLNAFMQPPEGLELVGLLAQGSARSRELAHAFGIPLYTS
PEQITRMPDIACIVVRSTVAGGTGTQLARHFLTRGVHVIQEHPLHPDDISSLQTLAQEQG
CCYWVNTFYPHTRAGRTWLRDAQQLRRCLAKTPPVVHATTSRQLLYSTLDLLLLALGVDA
AAVECDVVGSFSDFHCLRLFWPEGEACLLLQRYLDPDDPDMHSLIMHRLLLGWPEGHLSL
EASYGPVIWSSSLFVADHQENAHSLYRRPEILRDLPGLTRSAAPLSWRDCCETVGPEGVS
WLLHQLRSHLAGEHPPAACQSVHQIALSRLWQQILRKTGNAEIRRLTPPHHDRLAGFYND
DDKEAL


## Step 8 (exploratory) — Structural homolog check

A real experimental DsbA structure exists (PDB 4MCU), so before turning to ColabFold, a quick BLAST of the DsbA query against a single reference chain (4MCU, chain A) was used to check whether direct homology modeling from this template was viable.

In [43]:
# Save both sequences
with open('dsba_query.fasta', 'w') as f:
    f.write(">YP_005224660.1\nMSGHKKYLSVALLSAVLLPAMASAADAPATFTPEQEAKIGKIAADYLVAHPEVLLQASQKLQQIQAEQQASAATQAVLKNAAVLTQDKNTPTYGPANGKVTVIEFFDYQCVYCSRLAPVMEQVIKAHPQTRFAFKEWPIFGGRWESSLEAAKTGLQIYQQKGADAYLAYHNGIYATGHNEGKLTTADIQQQAKKAGFDAKKAADVEPVLQSINDLAQEIGLSGTPGVIVMPTTGATEASITVFPGLADKASLEAAIKKAGG")

with open('4mcu_chainA.fasta', 'w') as f:
    f.write(">4MCU_A\nSNAQITDGKQYITLDKPIAGEPQVLEFFSFYCPHCYQFEEVLHVSDNVRQKLPEGTKMTKYHVEFLGPLGKDLTQAWAVAIALGVEDKITAPMFEAVQKTQTVQSVADIRKVFVDAGVKGEDYDAAWNSFVVKSLVAQQEKAAADLQLQGVPAMYVNGKYQLNPQGMDTSNMDVFVAQYADTVKQLVEKK")

In [44]:
# Build a tiny "database" from the 4MCU sequence, then BLAST your query against it
!makeblastdb -in 4mcu_chainA.fasta -dbtype prot -out 4mcu_db
!blastp -query dsba_query.fasta -db 4mcu_db -outfmt 6



Building a new DB, current time: 08/09/2026 11:04:58
New DB name:   /content/4mcu_db
New DB title:  4mcu_chainA.fasta
Sequence type: Protein
Keep MBits: T
Maximum file size: 1000000000B
Adding sequences from FASTA; added 1 sequences in 0.000317097 seconds.


YP_005224660.1	4MCU_A	34.615	26	17	0	95	120	17	42	1.86e-04	26.6
YP_005224660.1	4MCU_A	25.806	31	23	0	161	191	119	149	5.0	13.1


The alignment is weak and fragmented (percent identity ~35% over only 26 residues, second HSP e-value ~5.0 — consistent with noise, not a genuine match), so direct template-based homology modeling from 4MCU was judged not viable. Both targets were predicted with ColabFold instead (README, Section 2).

## Save final target files to Drive

In [45]:
import shutil
import os

output_folder = '/content/drive/MyDrive/Subtractive_Genomics_KP_Results'

# Ensure the output folder exists
os.makedirs(output_folder, exist_ok=True)

files_to_save_locally = [
    'matched_ids.txt',
    'top3_targets.fasta',
    'final_2_targets.fasta'
]

for file_name in files_to_save_locally:
    destination_path = os.path.join(output_folder, file_name)
    if os.path.exists(file_name):
        shutil.copy(file_name, destination_path)
        print(f"Copied '{file_name}' to '{destination_path}'")
    else:
        print(f"File '{file_name}' not found. Skipping copy.")

File 'matched_ids.txt' not found. Skipping copy.
Copied 'top3_targets.fasta' to '/content/drive/MyDrive/Subtractive_Genomics_KP_Results/top3_targets.fasta'
Copied 'final_2_targets.fasta' to '/content/drive/MyDrive/Subtractive_Genomics_KP_Results/final_2_targets.fasta'
